# Quant Skills — Lesson 3: Options Pricing and the Greeks

**Goal:** build the formula that prices every option on your chain — **Black-Scholes** — and see
exactly where **delta, gamma, theta and vega** come from.

**How to run it:** **Runtime → Run all**. No data download this time — everything is math.

**Lines you can edit** are marked `# <- EDIT`. Plug in real numbers from thinkorswim and compare
what the formula says to what the market's charging.

## Step 1 — Load the tools

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import log, sqrt, exp, erf, pi

def N(x):   # chance a standard normal value is below x
    return 0.5 * (1 + erf(x / sqrt(2)))

def n(x):   # height of the bell curve at x
    return exp(-x * x / 2) / sqrt(2 * pi)

print("Tools loaded.")

## Step 2 — The Black-Scholes formula

Every option price comes from five inputs:

| Input | Meaning |
|---|---|
| **S** | Stock price now |
| **K** | Strike |
| **T** | Time to expiration, in years (days ÷ 365) |
| **σ (sigma)** | Implied volatility |
| **r** | Interest rate |

The formula asks one question: *given how much this stock tends to move, what's the fair price to own
the right to buy (call) or sell (put) it at K?* The Greeks are just how that price changes when one input moves.

In [ ]:
def black_scholes(S, K, days, iv, r=0.04, kind="put"):
    T = days / 365
    d1 = (log(S / K) + (r + iv**2 / 2) * T) / (iv * sqrt(T))
    d2 = d1 - iv * sqrt(T)
    if kind == "call":
        price = S * N(d1) - K * exp(-r * T) * N(d2)
        delta = N(d1)
        theta = (-S * n(d1) * iv / (2 * sqrt(T)) - r * K * exp(-r * T) * N(d2)) / 365
        prob_itm = N(d2)
    else:
        price = K * exp(-r * T) * N(-d2) - S * N(-d1)
        delta = N(d1) - 1
        theta = (-S * n(d1) * iv / (2 * sqrt(T)) + r * K * exp(-r * T) * N(-d2)) / 365
        prob_itm = N(-d2)
    gamma = n(d1) / (S * iv * sqrt(T))
    vega = S * n(d1) * sqrt(T) / 100
    return {"price": price, "delta": delta, "gamma": gamma,
            "theta": theta, "vega": vega, "prob_itm": prob_itm}

print("Formula ready.")

## Step 3 — Price a real option

Pick an option from your thinkorswim chain and type its numbers below. Then compare the formula's
price to the **mark** (mid) on your chain. They should be close — if they're not, the IV you typed
is probably off.

In [ ]:
S    = 740      # <- EDIT: stock price now
K    = 720      # <- EDIT: strike
DAYS = 30       # <- EDIT: calendar days to expiration
IV   = 0.18     # <- EDIT: implied vol from the chain (0.18 = 18%)
KIND = "put"    # <- EDIT: "put" or "call"

g = black_scholes(S, K, DAYS, IV, kind=KIND)
print(f"{KIND.upper()}  strike {K}  ·  {DAYS} days  ·  IV {IV*100:.1f}%  ·  stock ${S}")
print(f"  Fair price:    ${g['price']:.2f}   (= ${g['price']*100:,.0f} per contract)")
print(f"  Delta:         {g['delta']:+.3f}")
print(f"  Gamma:         {g['gamma']:.4f}")
print(f"  Theta:         {g['theta']:+.3f} per day   (= ${g['theta']*100:+,.2f} per contract per day)")
print(f"  Vega:          {g['vega']:+.3f} per 1 vol point")
print(f"  Chance it finishes in the money: {g['prob_itm']:.1%}")

### What each Greek tells you

- **Delta** — how much the option moves when the stock moves $1. Also a rough chance of finishing in the money.
- **Gamma** — how fast delta itself changes. High gamma = the option's behavior changes fast.
- **Theta** — how much value the option loses each day just from time passing. Bad for buyers, income for sellers.
- **Vega** — how much the option gains if implied vol rises 1 point. Why options get cheaper after earnings ("IV crush").

## Step 4 — Find the 16-delta strike

In Lesson 2 you learned a strike at the edge of the 1-standard-deviation expected move is about **16 delta**.
This finds it directly — scanning every strike until the delta hits your target.

In [ ]:
TARGET_DELTA = 0.16   # <- EDIT: 0.16 = standard credit-spread short strike, 0.25 = more aggressive
STRIKE_STEP  = 1      # <- EDIT: strike spacing ($1 for SPY/QQQ, $5 for SPX)

strikes = np.arange(round(S * 0.6), round(S * 1.4), STRIKE_STEP)
put_k  = min(strikes[strikes < S], key=lambda k: abs(abs(black_scholes(S, k, DAYS, IV, kind="put")["delta"])  - TARGET_DELTA))
call_k = min(strikes[strikes > S], key=lambda k: abs(abs(black_scholes(S, k, DAYS, IV, kind="call")["delta"]) - TARGET_DELTA))

move = S * IV * sqrt(DAYS / 365)
print(f"Expected move over {DAYS} days at {IV*100:.0f}% IV: ±${move:,.2f}  →  ${S-move:,.2f} to ${S+move:,.2f}")
print(f"{TARGET_DELTA*100:.0f}-delta put strike:  {put_k}   (delta {black_scholes(S, put_k, DAYS, IV, kind='put')['delta']:+.3f})")
print(f"{TARGET_DELTA*100:.0f}-delta call strike: {call_k}   (delta {black_scholes(S, call_k, DAYS, IV, kind='call')['delta']:+.3f})")
print()
print("Notice the 16-delta strikes sit right around the edges of the expected move.")

## Step 5 — Price a whole credit spread

Now combine two options into the trade you actually place. This prices a **put credit spread**
using the 16-delta strike from Step 4 as the short leg.

In [ ]:
WIDTH = 5     # <- EDIT: spread width in points

short_k, long_k = put_k, put_k - WIDTH
short = black_scholes(S, short_k, DAYS, IV, kind="put")
long_ = black_scholes(S, long_k, DAYS, IV, kind="put")

credit = short["price"] - long_["price"]
max_loss = WIDTH - credit
net_delta = -short["delta"] + long_["delta"]
net_theta = -short["theta"] + long_["theta"]

print(f"Put credit spread: SELL {short_k} put / BUY {long_k} put  ·  {DAYS} days")
print(f"  Credit (max gain):  ${credit:.2f}   (${credit*100:,.0f} per spread)")
print(f"  Max loss:           ${max_loss:.2f}   (${max_loss*100:,.0f} per spread)")
print(f"  Credit as % of width: {credit/WIDTH:.0%}   (your rule: at least 20%)")
print(f"  Breakeven:          ${short_k - credit:,.2f}")
print(f"  Chance of max gain (both expire worthless): {1 - short['prob_itm']:.1%}")
print(f"  Chance of max loss (finishes below {long_k}):  {long_['prob_itm']:.1%}")
print(f"  Spread delta:       {net_delta:+.3f}   (you gain a little when the stock rises)")
print(f"  Spread theta:       ${net_theta*100:+,.2f} per spread per day   (time works for you)")

## Step 6 — Why time decay speeds up near expiration

This tracks the same at-the-money option every day from 60 days out to expiration, holding everything
else constant. Watch the line drop faster at the end — that's theta accelerating.
It's why credit spread sellers close around 7–21 days out, and why buyers of short-dated options bleed.

In [ ]:
days_left = np.arange(60, 0, -1)
values = [black_scholes(S, S, d, IV, kind="put")["price"] for d in days_left]

plt.figure(figsize=(10, 4.5))
plt.plot(60 - days_left, values, linewidth=2)
plt.title(f"At-the-money put value as expiration approaches (IV {IV*100:.0f}%)")
plt.xlabel("Days passed"); plt.ylabel("Option value ($ per share)")
plt.grid(alpha=0.3); plt.show()

for d in (60, 30, 14, 7, 2):
    th = black_scholes(S, S, d, IV, kind="put")["theta"]
    print(f"{d:>2} days left: losing ${-th*100:,.2f} per contract per day")

## Step 7 — Why short-dated options swing so violently

This plots **delta** across a range of stock prices for options at different times to expiration.
Close to expiration, delta flips from near 0 to near 1 over a tiny price move — that's **gamma**.

**This is the chart behind your "I cut it, then it recovered" trades.** On a 0–2 day option, a small
stock move can wipe out most of the value in an hour, then bring it right back. With 30 days left,
the same move barely changes anything.

In [ ]:
prices = np.linspace(S * 0.95, S * 1.05, 200)
plt.figure(figsize=(10, 4.5))
for d in (30, 7, 2):
    deltas = [black_scholes(p, S, d, IV, kind="call")["delta"] for p in prices]
    plt.plot(prices, deltas, label=f"{d} days left", linewidth=2)
plt.axvline(S, color="black", linewidth=0.8)
plt.title("Call delta vs. stock price — steeper curve = more violent swings")
plt.xlabel("Stock price"); plt.ylabel("Delta"); plt.legend(); plt.grid(alpha=0.3); plt.show()

## Your exercises

1. **Price a real option.** In Step 3, type the stock price, strike, days, and IV of an option on your
   thinkorswim chain. How close is the formula to the chain's mark?
2. **Check your strikes.** Run Step 4 with real SPY numbers. Does the 16-delta strike the formula finds
   match the one your chain shows near −0.16 delta?
3. **See IV's effect.** In Step 3, change `IV` from 0.18 to 0.30 and re-run Steps 3–5. What happens to the
   credit you collect? (This is why you sell when IV is high.)
4. **Feel gamma.** In Step 3, set `DAYS = 2` and re-run Steps 3 and 7. Compare theta and gamma to the 30-day version.

**Next lesson:** statistics — distributions, fat tails, and why crashes like April 2025 happen more
often than the bell curve says they should.